In [ ]:
import pandas as pd
from dataHub import dataHub
from nba_api.stats.library.parameters import Season 
from sqlalchemy import text
from sqlalchemy.dialects.postgresql.base import PGDialect
PGDialect._get_server_version_info = lambda *args: (9, 2)
dh = dataHub()

# Always comment these lines so you have to verify connections prior to executing
# from_con = dh.db_connect('postgre'); to_con = dh.db_connect('cockroach')
# from_con = dh.db_connect('cockroach'); to_con = dh.db_connect('postgre')

# DROP VIEWS

In [18]:
df_views = pd.read_sql("SELECT * FROM information_schema.views WHERE table_name LIKE '%%_vw' ORDER BY table_name", to_con)
df_views.to_csv('vw_backup.csv', index=False)

db_ex = to_con.connect()
for _, row in df_views.iterrows():
    db_ex.execute(text(f'DROP VIEW IF EXISTS {row["table_schema"]+"."+row["table_name"]} CASCADE'))
db_ex.commit()

# NBA

INJURIES

In [ ]:
# Query from from_com
injuries = pd.read_sql(f"SELECT * FROM nba.injuries WHERE game_date >= '{Season.current_season_year + '-06-01'}'", from_con)

# Delete from to_con
to_ex = to_con.connect()
to_ex.execute(text(f"DELETE FROM nba.injuries WHERE game_date >= '{Season.current_season_year + '-06-01'}'"))
to_ex.commit()

# Write to to_con
injuries.drop_duplicates().to_sql('injuries', to_con, schema='nba', index=False, if_exists='append')

KEY DATES

In [5]:
# Query from from_com
key_dates = pd.read_sql(f"SELECT * FROM nba.key_dates WHERE season = '{Season.current_season}'", from_con)

# Delete from to_con
to_ex = to_con.connect()
to_ex.execute(text(f"DELETE FROM nba.key_dates WHERE season = '{Season.current_season}'"))
to_ex.commit()

# Write to to_con
key_dates.drop_duplicates().to_sql('key_dates', to_con, schema='nba', index=False, if_exists='append')

5

LEAGUE GAME SCHEDULE

In [6]:
# Query from from_com
league_game_schedule = pd.read_sql(f"SELECT * FROM nba.league_game_schedule WHERE season = '{Season.current_season}'", from_con)

# Delete from to_con
to_ex = to_con.connect()
to_ex.execute(text(f"DELETE FROM nba.league_game_schedule WHERE season = '{Season.current_season}'"))
to_ex.commit()

# Write to to_con
league_game_schedule.drop_duplicates().to_sql('league_game_schedule', to_con, schema='nba', index=False, if_exists='append')

407

PLAYER BOX SCORE

In [ ]:
# Query from from_com
cur_season_game_ids = ', '.join(league_game_schedule['game_id'].astype('str'))
player_box_score = pd.read_sql(f"SELECT * FROM nba.player_box_score WHERE game_id IN ({cur_season_game_ids})", from_con)

# Delete from to_con
to_ex = to_con.connect()
to_ex.execute(text(f"DELETE FROM nba.player_box_score WHERE game_id IN ({cur_season_game_ids})"))
to_ex.commit()

# Write to to_con
player_box_score.drop_duplicates().to_sql('player_box_score', to_con, schema='nba', index=False, if_exists='append')

397

PLAYER INFO

In [18]:
player_info = pd.read_sql('SELECT * FROM nba.player_info', from_con)
player_info.drop_duplicates().to_sql('player_info', to_con, schema='nba', index=False, if_exists='replace')

106

PLAYER SEASON STATS

In [19]:
player_season_stats = pd.read_sql('SELECT * FROM nba.player_season_stats', from_con)
player_season_stats.drop_duplicates().to_sql('player_season_stats', to_con, schema='nba', index=False, if_exists='replace')

560

TEAM BOX SCORE

In [10]:
# Query from from_com
team_box_score = pd.read_sql(f"SELECT * FROM nba.team_box_score WHERE game_id IN ({cur_season_game_ids})", from_con)

# Delete from to_con
to_ex = to_con.connect()
to_ex.execute(text(f"DELETE FROM nba.team_box_score WHERE game_id IN ({cur_season_game_ids})"))
to_ex.commit()

# Write to to_con
team_box_score.drop_duplicates().to_sql('team_box_score', to_con, schema='nba', index=False, if_exists='append')

546

TEAM ROSTER

In [ ]:
# Query from from_com
team_roster = pd.read_sql(f"SELECT * FROM nba.team_roster WHERE season = '{Season.current_season}'", from_con)

# Delete from to_con
to_ex = to_con.connect()
to_ex.execute(text(f"DELETE FROM nba.team_roster WHERE season = '{Season.current_season}'"))
to_ex.commit()

# Write to to_con
team_roster.drop_duplicates().to_sql('team_roster', to_con, schema='nba', index=False, if_exists='append')

717

TEAMS

In [22]:
teams = pd.read_sql('SELECT * FROM nba.teams', from_con)
teams.drop_duplicates().to_sql('teams', to_con, schema='nba', index=False, if_exists='replace')

30

TRANSACTION LOG

In [13]:
# Query from from_com
transaction_log = pd.read_sql(f"SELECT * FROM nba.transaction_log WHERE date >= '{str(Season.current_season_year) + '-01-01'}'", from_con)

# Delete from to_con
to_ex = to_con.connect()
to_ex.execute(text(f"DELETE FROM nba.transaction_log WHERE date >= '{str(Season.current_season_year) + '-01-01'}'"))
to_ex.commit()

# Write to to_con
transaction_log.drop_duplicates().to_sql('transaction_log', to_con, schema='nba', index=False, if_exists='append')

657

# FTY

COMPETITOR ROSTER

In [14]:
# Query from from_com
competitor_roster = pd.read_sql(f"SELECT * FROM fty.competitor_roster WHERE season = '{Season.current_season}'", from_con)

# Delete from to_con
to_ex = to_con.connect()
to_ex.execute(text(f"DELETE FROM fty.competitor_roster WHERE season = '{Season.current_season}'"))
to_ex.commit()

# Write to to_con
competitor_roster.drop_duplicates().to_sql('competitor_roster', to_con, schema='fty', index=False, if_exists='append')

434

FREE AGENTS

In [25]:
free_agents = pd.read_sql('SELECT * FROM fty.free_agents', from_con)
free_agents.to_sql('free_agents', to_con, schema='fty', index=False, if_exists='replace')

442

LEAGUE

In [16]:
# Query from from_com
league = pd.read_sql(f"SELECT * FROM fty.league WHERE season = '{Season.current_season}'", from_con)

# Delete from to_con
to_ex = to_con.connect()
to_ex.execute(text(f"DELETE FROM fty.league WHERE season = '{Season.current_season}'"))
to_ex.commit()

# Write to to_con
league.drop_duplicates().to_sql('league', to_con, schema='fty', index=False, if_exists='append')

2

LEAGUE COMPETITOR

In [17]:
# Query from from_com
league_competitor = pd.read_sql(f"SELECT * FROM fty.league_competitor WHERE season = '{Season.current_season}'", from_con)

# Delete from to_con
to_ex = to_con.connect()
to_ex.execute(text(f"DELETE FROM fty.league_competitor WHERE season = '{Season.current_season}'"))
to_ex.commit()

# Write to to_con
league_competitor.drop_duplicates().to_sql('league_competitor', to_con, schema='fty', index=False, if_exists='append')

20

LEAGUE SCHEDULE

In [18]:
# Query from from_com
league_schedule = pd.read_sql(f"SELECT * FROM fty.league_schedule WHERE season = '{Season.current_season}'", from_con)

# Delete from to_con
to_ex = to_con.connect()
to_ex.execute(text(f"DELETE FROM fty.league_schedule WHERE season = '{Season.current_season}'"))
to_ex.commit()

# Write to to_con
league_schedule.drop_duplicates().to_sql('league_schedule', to_con, schema='fty', index=False, if_exists='append')

396

MATCHUP BOX SCORE

In [19]:
# Query from from_com
matchup_box_score = pd.read_sql(f"SELECT * FROM fty.matchup_box_score WHERE season = '{Season.current_season}'", from_con)

# Delete from to_con
to_ex = to_con.connect()
to_ex.execute(text(f"DELETE FROM fty.matchup_box_score WHERE season = '{Season.current_season}'"))
to_ex.commit()

# Write to to_con
matchup_box_score.drop_duplicates().to_sql('matchup_box_score', to_con, schema='fty', index=False, if_exists='append')

400

RECENT ACTIVITY

In [20]:
# Query from from_com
recent_activity = pd.read_sql(f"SELECT * FROM fty.recent_activity WHERE season = '{Season.current_season}'", from_con)

# Delete from to_con
to_ex = to_con.connect()
to_ex.execute(text(f"DELETE FROM fty.recent_activity WHERE season = '{Season.current_season}'"))
to_ex.commit()

# Write to to_con
recent_activity.drop_duplicates().to_sql('recent_activity', to_con, schema='fty', index=False, if_exists='append')

12

# UTIL

In [5]:
nba_fty_name_match = pd.read_sql('SELECT * FROM util.nba_fty_name_match', from_con)
nba_fty_name_match.to_sql('nba_fty_name_match', to_con, schema='util', index=False, if_exists='replace')

28

In [6]:
table_column_order = pd.read_sql('SELECT * FROM util.table_column_order', from_con)
table_column_order.to_sql('table_column_order', to_con, schema='util', index=False, if_exists='replace')

219

In [32]:
update_log = pd.read_sql('SELECT * FROM util.update_log', from_con)
update_log.to_sql('update_log', to_con, schema='util', index=False, if_exists='replace')

927

In [33]:
update_schedule = pd.read_sql('SELECT * FROM util.update_schedule', from_con)
update_schedule.to_sql('update_schedule', to_con, schema='util', index=False, if_exists='replace')

14

# ANL

In [34]:
pts_prediction = pd.read_sql('SELECT * FROM anl.pts_prediction', from_con)
pts_prediction.drop_duplicates().to_sql('pts_prediction', to_con, schema='anl', index=False, if_exists='replace')

929

# RECREATE VIEWS

In [4]:
db_ex = to_con.connect()
for _, row in df_views.iterrows():
    db_ex.execute(text(f'CREATE VIEW {row["table_schema"]+"."+row["table_name"]} AS {row['view_definition']}'))
db_ex.commit()